In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime

DB_FILE = "cyber_cafe.db"

# ==========================================
# DATABASE INITIALIZATION (AUTOMATIC)
# ==========================================
def init_db():
    try:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()

        # Enable Foreign Keys
        cursor.execute("PRAGMA foreign_keys = ON;")

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS PCs (
                pc_id INTEGER PRIMARY KEY AUTOINCREMENT,
                pc_name TEXT NOT NULL,
                rate_per_hour REAL NOT NULL,
                status TEXT CHECK(status IN ('Available', 'Occupied')) DEFAULT 'Available'
            );
        """)

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Customers (
                customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
                full_name TEXT NOT NULL,
                phone_number TEXT UNIQUE NOT NULL
            );
        """)

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Sessions (
                session_id INTEGER PRIMARY KEY AUTOINCREMENT,
                pc_id INTEGER NOT NULL,
                customer_id INTEGER NOT NULL,
                start_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                end_time TIMESTAMP NULL,
                total_bill REAL NULL,
                payment_status TEXT CHECK(payment_status IN ('Active', 'Completed')) DEFAULT 'Active',
                FOREIGN KEY (pc_id) REFERENCES PCs(pc_id),
                FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
            );
        """)

        # Auto-seed sample PCs if empty
        cursor.execute("SELECT COUNT(*) FROM PCs")
        if cursor.fetchone()[0] == 0:
            default_pcs = [
                ('PC-01 (VIP)', 80.00, 'Available'),
                ('PC-02 (VIP)', 80.00, 'Available'),
                ('PC-03 (Gaming)', 50.00, 'Available'),
                ('PC-04 (Gaming)', 50.00, 'Available'),
                ('PC-05 (Budget)', 30.00, 'Available'),
                ('PC-06 (Budget)', 30.00, 'Available')
            ]
            cursor.executemany("INSERT INTO PCs (pc_name, rate_per_hour, status) VALUES (?, ?, ?)", default_pcs)
            conn.commit()

        conn.close()
        return True
    except sqlite3.Error as err:
        messagebox.showerror("Database Error", f"SQLite Initialization Failed: {err}")
        return False

# ==========================================
# MAIN APPLICATION CLASS
# ==========================================
class CyberCafeApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Cyber Café / Gaming Parlor PC Allocator")
        self.root.geometry("900x650")
        self.root.resizable(False, False)

        self.style = ttk.Style()
        self.style.theme_use("clam")

        self.setup_ui()
        if init_db():
            self.refresh_data()

    def setup_ui(self):
        header = tk.Label(
            self.root,
            text="💻 CYBER CAFÉ & GAMING PC ALLOCATION MANAGEMENT",
            font=("Helvetica", 16, "bold"),
            bg="#2c3e50",
            fg="white",
            pady=10
        )
        header.pack(fill=tk.X)

        main_container = ttk.Frame(self.root, padding=10)
        main_container.pack(fill=tk.BOTH, expand=True)

        left_frame = ttk.LabelFrame(main_container, text=" Live PC Status Grid ", padding=10)
        left_frame.place(x=0, y=0, width=420, height=550)

        self.grid_container = tk.Frame(left_frame)
        self.grid_container.pack(fill=tk.BOTH, expand=True)

        right_frame = ttk.Frame(main_container)
        right_frame.place(x=430, y=0, width=440, height=550)

        # 1. Start Session Form
        start_box = ttk.LabelFrame(right_frame, text=" Start New Session (Allocation) ", padding=10)
        start_box.pack(fill=tk.X, pady=(0, 10))

        ttk.Label(start_box, text="Customer Name:").grid(row=0, column=0, sticky=tk.W, pady=5)
        self.entry_cust_name = ttk.Entry(start_box, width=25)
        self.entry_cust_name.grid(row=0, column=1, pady=5)

        ttk.Label(start_box, text="Phone Number:").grid(row=1, column=0, sticky=tk.W, pady=5)
        self.entry_cust_phone = ttk.Entry(start_box, width=25)
        self.entry_cust_phone.grid(row=1, column=1, pady=5)

        ttk.Label(start_box, text="Select Free PC:").grid(row=2, column=0, sticky=tk.W, pady=5)
        self.combo_pcs = ttk.Combobox(start_box, state="readonly", width=23)
        self.combo_pcs.grid(row=2, column=1, pady=5)

        btn_start = tk.Button(
            start_box,
            text="▶ Allocate PC & Start Session",
            bg="#27ae60",
            fg="white",
            font=("Helvetica", 9, "bold"),
            command=self.start_session
        )
        btn_start.grid(row=3, column=0, columnspan=2, pady=10, sticky="ew")

        # 2. Active Sessions Table
        table_box = ttk.LabelFrame(right_frame, text=" Active Sessions (End Session / Checkout) ", padding=10)
        table_box.pack(fill=tk.BOTH, expand=True)

        cols = ("Session ID", "PC", "Customer", "Start Time")
        self.tree = ttk.Treeview(table_box, columns=cols, show="headings", height=8)
       
        self.tree.heading("Session ID", text="ID")
        self.tree.heading("PC", text="PC")
        self.tree.heading("Customer", text="Customer")
        self.tree.heading("Start Time", text="Start Time")

        self.tree.column("Session ID", width=40, anchor=tk.CENTER)
        self.tree.column("PC", width=80, anchor=tk.CENTER)
        self.tree.column("Customer", width=120)
        self.tree.column("Start Time", width=120, anchor=tk.CENTER)

        self.tree.pack(fill=tk.BOTH, expand=True, pady=5)

        btn_end = tk.Button(
            table_box,
            text="⏹ End Session & Generate Bill",
            bg="#c0392b",
            fg="white",
            font=("Helvetica", 9, "bold"),
            command=self.end_session
        )
        btn_end.pack(fill=tk.X, pady=5)

    def refresh_data(self):
        for widget in self.grid_container.winfo_children():
            widget.destroy()

        for item in self.tree.get_children():
            self.tree.delete(item)

        try:
            conn = sqlite3.connect(DB_FILE)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()

            pc_query = """
                SELECT p.pc_id, p.pc_name, p.rate_per_hour, p.status, c.full_name AS active_user
                FROM PCs p
                LEFT JOIN Sessions s ON p.pc_id = s.pc_id AND s.payment_status = 'Active'
                LEFT JOIN Customers c ON s.customer_id = c.customer_id
                ORDER BY p.pc_id ASC
            """
            cursor.execute(pc_query)
            pcs = cursor.fetchall()

            available_pcs = []
            for index, pc in enumerate(pcs):
                if pc["status"] == "Available":
                    color = "#2ecc71"
                    card_text = f"{pc['pc_name']}\n₹{pc['rate_per_hour']}/hr\n[ FREE ]"
                    available_pcs.append(f"{pc['pc_id']}: {pc['pc_name']} (₹{pc['rate_per_hour']}/hr)")
                else:
                    color = "#e74c3c"
                    user_name = pc['active_user'] if pc['active_user'] else "Occupied"
                    card_text = f"{pc['pc_name']}\n👤 {user_name}\n[ IN USE ]"

                card = tk.Label(
                    self.grid_container,
                    text=card_text,
                    bg=color,
                    fg="white",
                    font=("Helvetica", 10, "bold"),
                    width=18,
                    height=4,
                    relief=tk.RAISED,
                    bd=2
                )
                row = index // 2
                col = index % 2
                card.grid(row=row, column=col, padx=8, pady=8)

            self.combo_pcs["values"] = available_pcs
            if available_pcs:
                self.combo_pcs.current(0)
            else:
                self.combo_pcs.set("No PCs Available")

            session_query = """
                SELECT s.session_id, p.pc_name, c.full_name, s.start_time
                FROM Sessions s
                JOIN PCs p ON s.pc_id = p.pc_id
                JOIN Customers c ON s.customer_id = c.customer_id
                WHERE s.payment_status = 'Active'
            """
            cursor.execute(session_query)
            active_sessions = cursor.fetchall()

            for session in active_sessions:
                self.tree.insert("", tk.END, values=(
                    session["session_id"],
                    session["pc_name"],
                    session["full_name"],
                    session["start_time"]
                ))

            conn.close()
        except sqlite3.Error as err:
            messagebox.showerror("Database Error", str(err))

    def start_session(self):
        name = self.entry_cust_name.get().strip()
        phone = self.entry_cust_phone.get().strip()
        selected_pc = self.combo_pcs.get()

        if not name or not phone:
            messagebox.showwarning("Input Error", "Please enter Customer Name and Phone Number.")
            return

        if "No PCs Available" in selected_pc or not selected_pc:
            messagebox.showwarning("Allocation Error", "No available PC selected.")
            return

        pc_id = int(selected_pc.split(":")[0])

        try:
            conn = sqlite3.connect(DB_FILE)
            cursor = conn.cursor()

            cursor.execute("SELECT customer_id FROM Customers WHERE phone_number = ?", (phone,))
            result = cursor.fetchone()

            if result:
                customer_id = result[0]
            else:
                cursor.execute("INSERT INTO Customers (full_name, phone_number) VALUES (?, ?)", (name, phone))
                customer_id = cursor.lastrowid

            start_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cursor.execute("INSERT INTO Sessions (pc_id, customer_id, start_time) VALUES (?, ?, ?)", (pc_id, customer_id, start_time_str))
            cursor.execute("UPDATE PCs SET status = 'Occupied' WHERE pc_id = ?", (pc_id,))

            conn.commit()
            conn.close()

            messagebox.showinfo("Success", f"Session started for {name}!")

            self.entry_cust_name.delete(0, tk.END)
            self.entry_cust_phone.delete(0, tk.END)

        except sqlite3.Error as err:
            messagebox.showerror("Database Error", str(err))

        self.refresh_data()

    def end_session(self):
        selected_item = self.tree.selection()
        if not selected_item:
            messagebox.showwarning("Selection Error", "Please select an active session from the table to checkout.")
            return

        session_id = self.tree.item(selected_item)["values"][0]

        try:
            conn = sqlite3.connect(DB_FILE)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()

            query = """
                SELECT s.session_id, s.pc_id, s.start_time, p.rate_per_hour, c.full_name, p.pc_name
                FROM Sessions s
                JOIN PCs p ON s.pc_id = p.pc_id
                JOIN Customers c ON s.customer_id = c.customer_id
                WHERE s.session_id = ?
            """
            cursor.execute(query, (session_id,))
            session = cursor.fetchone()

            if not session:
                conn.close()
                return

            end_time = datetime.now()
            start_time = datetime.strptime(session["start_time"], "%Y-%m-%d %H:%M:%S")
           
            duration_seconds = (end_time - start_time).total_seconds()
            duration_hours = max(duration_seconds / 3600.0, 1 / 60.0)
           
            rate = float(session["rate_per_hour"])
            total_bill = round(duration_hours * rate, 2)

            cursor.execute(
                "UPDATE Sessions SET end_time = ?, total_bill = ?, payment_status = 'Completed' WHERE session_id = ?",
                (end_time.strftime("%Y-%m-%d %H:%M:%S"), total_bill, session_id)
            )
            cursor.execute("UPDATE PCs SET status = 'Available' WHERE pc_id = ?", (session["pc_id"],))

            conn.commit()
            conn.close()

            receipt = (
                f"🧾 --- BILL RECEIPT ---\n\n"
                f"Customer: {session['full_name']}\n"
                f"PC Used: {session['pc_name']}\n"
                f"Rate: ₹{rate}/hr\n"
                f"Duration: {round(duration_seconds/60, 1)} minutes\n"
                f"-----------------------------\n"
                f"Total Bill: ₹{total_bill:.2f}"
            )
            messagebox.showinfo("Session Completed", receipt)

        except sqlite3.Error as err:
            messagebox.showerror("Database Error", str(err))

        self.refresh_data()

if __name__ == "__main__":
    root = tk.Tk()
    app = CyberCafeApp(root)
    root.mainloop()

***